In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import Qwen3Model

from src.data import JLMTokenizer, JLMDataset, jlm_collate_fn
from src.model import BilinearDecisionHead


device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:

raw_dataset = load_dataset("tasksource/tasksource-jev-typed-decisions", split="train")


dataset = raw_dataset.filter(lambda example: example["kind"] == "choice")


train_dataset_sample = dataset.select(range(200)) 
print(f"Total examples: {len(train_dataset_sample)}")

In [ ]:
tokenizer = JLMTokenizer("Qwen/Qwen3-0.6B")
model = JLM("Qwen/Qwen3-0.6B")

model.base_model.eval()
for param in model.base_model.parameters():
    param.requires_grad = False

model.head.train()
for param in model.head.parameters():
    param.requires_grad = True

In [ ]:

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.head.parameters(), lr=1e-3)
train_data = JLMDataset(train_dataset_sample, tokenizer)

train_loader = DataLoader(
    train_data, 
    batch_size=4, 
    shuffle=True, 
    collate_fn=jlm_collate_fn
)

num_epochs = 3

print("--- Training Started ---")
for epoch in range(num_epochs):
    total_loss = 0.0
    correct = 0
    total = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        position_ids = batch["position_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        choice_idx = batch["choices_end_indxs"]
        answer_idx = batch["answer_end_indxs"]

        # Forward pass 
        logits_list = model(
            input_ids=input_ids,
            position_ids=position_ids,
            attention_mask=attention_mask,
            choice_idx=choice_idx,
            answer_idx=answer_idx
        )

        
        batch_loss = torch.tensor(0.0, device=device, requires_grad=True)
        
        for i, logits in enumerate(logits_list):
            item_logits = logits.unsqueeze(0)       # (1, num_choices)
            item_label = labels[i].unsqueeze(0)      # (1,)

            loss_i = criterion(item_logits, item_label)
            batch_loss = batch_loss + loss_i

            
            pred = torch.argmax(item_logits, dim=-1)
            if pred == item_label:
                correct += 1
            total += 1

        batch_loss = batch_loss / len(logits_list)

        # Backpropagation
        batch_loss.backward()
        optimizer.step()

        total_loss += batch_loss.item()

    avg_loss = total_loss / len(train_loader)
    acc = (correct / total) * 100
    print(f"Epoch {epoch + 1}/{num_epochs} | Loss: {avg_loss:.4f} | Accuracy: {acc:.2f}%")

print("--- Entrenamiento finalizado ---")